# KnobNet — MERT Layer 실험
MERT-v1-95M의 어느 레이어 출력이 노브 추정에 최적인지 비교합니다.

**실험 조건**
- 레이어: 1 ~ 12
- Phase 1: 20 epoch (MERT frozen)
- Phase 2: 3 epoch (MERT unfreeze)
- **중단 후 재실행 시 자동으로 이어서 진행**

## 1. 환경 설치

In [ ]:
!pip install -q transformers soundfile torchaudio

## 2. GitHub 클론

In [ ]:
import os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/KnobNet.git"  # <-- 수정
PROJECT_DIR = "/content/KnobNet"

if not os.path.exists(PROJECT_DIR):
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print("현재 디렉터리:", os.getcwd())

In [ ]:
# ── 테스트 모드 설정 ──────────────────────────────────────────────────────────
# True  → smallTest.zip 사용 / 레이어 [1, 12] / 각 Phase 1 epoch (파이프라인 검증용)
# False → 전체 데이터 / 레이어 1~12 / Phase1=60ep, Phase2=3ep

TEST_MODE = False

## 3. Google Drive 마운트 & 데이터 압축 해제

In [ ]:
from pathlib import Path

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)
!apt-get install -q p7zip-full

if TEST_MODE:
    zip_file = "/content/drive/MyDrive/KnobNet/smallTest.zip"
    print(f"[TEST] 압축 해제: {zip_file}")
    !7z x "{zip_file}" -o"{DATA_DIR}" -y
    print("완료")
else:
    ZIP_FILES = [
        "/content/drive/MyDrive/KnobNet/data.zip.001",
    ]
    for zip_path in ZIP_FILES:
        zip_path = Path(zip_path)
        if not zip_path.exists():
            print(f"[SKIP] 파일 없음: {zip_path}")
            continue
        print(f"압축 해제 중: {zip_path.name} ...")
        !7z x "{zip_path}" -o"{DATA_DIR}" -y
        print("완료")

In [ ]:
from pathlib import Path

ZIP_FILES = [
    "/content/drive/MyDrive/KnobNet/data.zip.001",
]

DATA_DIR = Path(PROJECT_DIR) / "data"
DATA_DIR.mkdir(exist_ok=True)
!apt-get install -q p7zip-full

for zip_path in ZIP_FILES:
    zip_path = Path(zip_path)
    if not zip_path.exists():
        print(f"[SKIP] 파일 없음: {zip_path}")
        continue
    print(f"압축 해제 중: {zip_path.name} ...")
    !7z x "{zip_path}" -o"{DATA_DIR}" -y
    print("완료")

import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

if TEST_MODE:
    # smallTest.zip 구조: input/ + output/black/ (csv + wav)
    train_loader, val_loader = make_loaders(
        dataset_root = PROJECT_DIR,
        wet_dir      = "data/output/black",
        batch_size   = 4,
        val_split    = 0.2,
        num_workers  = 0,
    )
    print(f"[TEST] train: {len(train_loader.dataset):,}  val: {len(val_loader.dataset):,}")
else:
    train_loader, val_loader = make_loaders(
        dataset_root = PROJECT_DIR,
        wet_dir      = "data/wet/black",
        batch_size   = 16,
        val_split    = 0.2,
        num_workers  = 2,
    )
    print(f"train: {len(train_loader.dataset):,}  val: {len(val_loader.dataset):,}")

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from dataset.loader import make_loaders

train_loader, val_loader = make_loaders(
    dataset_root = PROJECT_DIR,
    wet_dir      = "data/wet/black",
    batch_size   = 16,
    val_split    = 0.2,
    num_workers  = 2,
)
print(f"train: {len(train_loader.dataset):,}  val: {len(val_loader.dataset):,}")

## 5. 실험 설정

In [ ]:
LAYERS        = list(range(1, 13))   # 1 ~ 12
PHASE1_EPOCHS = 60
PHASE2_EPOCHS = 3

if TEST_MODE:
    LAYERS        = [1, 12]
    PHASE1_EPOCHS = 1
    PHASE2_EPOCHS = 1

CKPT_ROOT = Path("/content/drive/MyDrive/KnobNet/layer_exp" + ("_test" if TEST_MODE else ""))
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"{'[TEST MODE]' if TEST_MODE else '[FULL MODE]'}")
print(f"레이어: {LAYERS}")
print(f"Phase1={PHASE1_EPOCHS}ep  Phase2={PHASE2_EPOCHS}ep")
print(f"체크포인트: {CKPT_ROOT}")

## 6. 헬퍼 함수 (로깅 + resume)

In [ ]:
import csv
import torch
import torch.nn as nn

from model.model import KnobNet
from utils.config import KNOB_PARAMS
from train.train import (
    run_epoch, evaluate_all,
    make_optimizer, load_checkpoint,
)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.L1Loss()
TOLERANCE = 0.1
print("device:", device)


def log_to_csv(csv_path, row: dict):
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def print_epoch(layer_idx, phase, epoch, total_ep, train_loss, metrics, improved):
    mark = " *" if improved else ""
    print(f"[layer={layer_idx:2d} | P{phase} ep{epoch:02d}/{total_ep}]  "
          f"train={train_loss:.4f}  val={metrics['val_loss']:.4f}{mark}")
    print("  MAE  " + "  ".join(f"{k}={v:.4f}" for k, v in metrics["mae"].items()))
    print("  Acc  " + "  ".join(f"{k}={v*100:.1f}%" for k, v in metrics["acc"].items()))


def is_layer_done(log_csv, phase2_epochs):
    """CSV 로그로 레이어 완료 여부 확인 (checkpoint 없어도 OK)"""
    if not log_csv.exists():
        return False
    with open(log_csv, newline="") as f:
        for row in csv.DictReader(f):
            if int(row["phase"]) == 2 and int(row["epoch"]) >= phase2_epochs:
                return True
    return False


def read_best_from_csv(log_csv, phase=2):
    """CSV에서 해당 phase의 val_loss 최소 행을 읽어 results 형식으로 반환"""
    with open(log_csv, newline="") as f:
        rows = [r for r in csv.DictReader(f) if int(r["phase"]) == phase]
    best = min(rows, key=lambda r: float(r["val_loss"]))
    return {
        "val_loss": float(best["val_loss"]),
        "mae": {k: float(best[f"mae_{k}"]) for k in KNOB_PARAMS},
        "acc": {**{k: float(best[f"acc_{k}"]) for k in KNOB_PARAMS},
                "all": float(best["acc_all"])},
    }


def run_phase(layer_idx, phase, epochs, model, log_csv, ckpt_latest):
    """한 phase 학습. ckpt_latest (레이어당 1개) 에서 자동 resume."""
    optimizer = make_optimizer(model, lr=1e-3, phase=phase)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    start_epoch = 1
    best_val    = float("inf")

    if ckpt_latest.exists():
        meta = torch.load(ckpt_latest, map_location="cpu")
        if meta["phase"] == phase:
            # 같은 phase 중단 후 재개
            meta = load_checkpoint(ckpt_latest, model, optimizer, scheduler)
            start_epoch = meta["epoch"] + 1
            best_val    = meta.get("best_val", float("inf"))
            print(f"  [resume] layer={layer_idx} Phase{phase} epoch {meta['epoch']} → {start_epoch}부터 재개")
        else:
            # phase 전환 (1→2): latest에서 model weights만 복원
            load_checkpoint(ckpt_latest, model)

    if start_epoch > epochs:
        print(f"  [skip] layer={layer_idx} Phase{phase} 이미 완료")
        return

    for epoch in range(start_epoch, epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer, criterion, device, scaler)
        metrics    = evaluate_all(model, val_loader, criterion, device, tolerance=TOLERANCE)
        val_loss   = metrics["val_loss"]
        scheduler.step()

        improved = val_loss < best_val
        if improved:
            best_val = val_loss

        # latest: 레이어당 1개, 매 epoch 덮어씀 (resume용)
        torch.save({
            "phase": phase, "epoch": epoch, "best_val": best_val,
            "model_state":     model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "val_loss":        val_loss,
        }, ckpt_latest)

        print_epoch(layer_idx, phase, epoch, epochs, train_loss, metrics, improved)
        log_to_csv(log_csv, {
            "layer": layer_idx, "phase": phase, "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss":   round(val_loss, 6),
            **{f"mae_{k}": round(v, 6) for k, v in metrics["mae"].items()},
            **{f"acc_{k}": round(v, 6) for k, v in metrics["acc"].items()},
        })


print("헬퍼 함수 정의 완료")

## 7. 레이어별 학습 루프

In [ ]:
results = {}

for layer_idx in LAYERS:
    print(f"\n{'='*60}")
    print(f"  Layer {layer_idx:2d} / 12")
    print(f"{'='*60}")

    ckpt_latest = CKPT_ROOT / f"layer{layer_idx:02d}_latest.pt"
    log_csv     = CKPT_ROOT / f"layer{layer_idx:02d}_log.csv"

    # ── 이미 완료된 경우: CSV에서 결과 읽기 ──────────────────────────────
    if is_layer_done(log_csv, PHASE2_EPOCHS):
        print(f"  [skip] layer={layer_idx} 모든 Phase 완료 → CSV에서 로드")
        results[layer_idx] = read_best_from_csv(log_csv, phase=2)
        continue

    # ── 모델 초기화 ───────────────────────────────────────────────────────
    model = KnobNet(num_knobs=len(KNOB_PARAMS), freeze_mert=True,
                    layer_idx=layer_idx).to(device)

    # ── Phase 1: MERT frozen ──────────────────────────────────────────────
    phase1_done = (
        ckpt_latest.exists() and
        torch.load(ckpt_latest, map_location="cpu")["phase"] == 1 and
        torch.load(ckpt_latest, map_location="cpu")["epoch"] >= PHASE1_EPOCHS
    )
    if not phase1_done:
        print(f"  Phase 1  (MERT frozen, {PHASE1_EPOCHS} epoch)")
        run_phase(layer_idx, 1, PHASE1_EPOCHS, model, log_csv, ckpt_latest)
    else:
        print(f"  [skip] Phase 1 완료")
        load_checkpoint(ckpt_latest, model)

    # ── Phase 2: MERT unfreeze ────────────────────────────────────────────
    print(f"  Phase 2  (MERT unfreeze, {PHASE2_EPOCHS} epoch)")
    model.unfreeze_mert()
    run_phase(layer_idx, 2, PHASE2_EPOCHS, model, log_csv, ckpt_latest)

    # ── 완료: checkpoint 삭제, 결과는 CSV에서 ────────────────────────────
    if ckpt_latest.exists():
        ckpt_latest.unlink()
        print(f"  [cleanup] {ckpt_latest.name} 삭제 완료")

    results[layer_idx] = read_best_from_csv(log_csv, phase=2)
    print(f"  [Layer {layer_idx}] 최종 MAE  "
          + "  ".join(f"{k}={v:.4f}" for k, v in results[layer_idx]["mae"].items()))
    print(f"  [Layer {layer_idx}] 최종 Acc  "
          + "  ".join(f"{k}={v*100:.1f}%" for k, v in results[layer_idx]["acc"].items()))

print("\n\n모든 레이어 실험 완료")

## 8. 결과 비교 테이블 & 그래프

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for layer_idx, m in sorted(results.items()):
    rows.append({
        "layer":     layer_idx,
        "drive_MAE": round(m["mae"]["drive"], 4),
        "level_MAE": round(m["mae"]["level"], 4),
        "tone_MAE":  round(m["mae"]["tone"],  4),
        "avg_MAE":   round(sum(m["mae"].values()) / len(m["mae"]), 4),
        "drive_Acc": round(m["acc"]["drive"], 3),
        "level_Acc": round(m["acc"]["level"], 3),
        "tone_Acc":  round(m["acc"]["tone"],  3),
        "avg_Acc":   round((m["acc"]["drive"]+m["acc"]["level"]+m["acc"]["tone"])/3, 3),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# CSV 저장
csv_path = CKPT_ROOT / "layer_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nCSV 저장: {csv_path}")

# 그래프
layers = df["layer"].tolist()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for col, label in [("drive_MAE","drive"),("level_MAE","level"),("tone_MAE","tone"),("avg_MAE","avg")]:
    ax1.plot(layers, df[col], marker="o", linestyle="--" if "avg" in col else "-", label=label)
ax1.set_xlabel("MERT Layer"); ax1.set_ylabel("MAE")
ax1.set_title("MAE vs MERT Layer"); ax1.set_xticks(layers)
ax1.legend(); ax1.grid(True, alpha=0.3)

for col, label in [("drive_Acc","drive"),("level_Acc","level"),("tone_Acc","tone"),("avg_Acc","avg")]:
    ax2.plot(layers, df[col], marker="o", linestyle="--" if "avg" in col else "-", label=label)
ax2.set_xlabel("MERT Layer"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy vs MERT Layer"); ax2.set_xticks(layers)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = CKPT_ROOT / "layer_results.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"그래프 저장: {fig_path}")